# 08 — Construction d'Équipe Optimale en fonction des types demandé


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import requests
from math import pi

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F5F6FA',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

STATS = ['hp', 'attack', 'defense', 'special-attack', 'special-defense', 'speed']
TYPE_COLORS = {
    'normal': '#A8A878', 'fire': '#F08030', 'water': '#6890F0', 'electric': '#F8D030',
    'grass': '#78C850', 'ice': '#98D8D8', 'fighting': '#C03028', 'poison': '#A040A0',
    'ground': '#E0C068', 'flying': '#A890F0', 'psychic': '#F85888', 'bug': '#A8B820',
    'rock': '#B8A038', 'ghost': '#705898', 'dragon': '#7038F8', 'dark': '#705848',
    'steel': '#B8B8D0', 'fairy': '#EE99AC', 'stellar': '#40B5A5',
}
TEAM_COLORS = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C']

with open('../data/pokemon_full.pkl', 'rb') as f:
    data = pickle.load(f)
df, matrix, ALL_TYPES = data['df'], data['matrix'], data['ALL_TYPES']
print(f'Données chargées : {df.shape[0]} Pokémon')

df['primary_type'] = df['types'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x)

for type_name, count in df['primary_type'].value_counts().items():
    print(f"  {type_name} : {count} Pokémon")

Données chargées : 1025 Pokémon
  water : 134 Pokémon
  normal : 118 Pokémon
  grass : 103 Pokémon
  bug : 83 Pokémon
  fire : 66 Pokémon
  psychic : 60 Pokémon
  electric : 59 Pokémon
  rock : 58 Pokémon
  dark : 45 Pokémon
  poison : 42 Pokémon
  ground : 40 Pokémon
  fighting : 40 Pokémon
  dragon : 37 Pokémon
  steel : 36 Pokémon
  ghost : 35 Pokémon
  ice : 31 Pokémon
  fairy : 29 Pokémon
  flying : 9 Pokémon


## Configuration

In [ ]:
TEAM_SIZE      = 6
REQUIRED_TYPE  = 'water'
REQUIRED_COUNT = 3
N_ITER         = 40000
POOL_REQUIRED  = 180
POOL_OTHER     = 320
SEED           = 42

def has_type(type_list, t):
    if isinstance(type_list, list): return t in type_list
    if isinstance(type_list, str):  return type_list == t
    return False

def build_pools(df):
    work = df.copy()
    work['is_required_type'] = work['types'].apply(lambda x: has_type(x, REQUIRED_TYPE))
    work['solo_score'] = (
        work['bst']
        + 10 * work['defense_score']
        +  4 * work['n_resistances']
        +  8 * work['n_immunities']
        +  3 * work['n_offense']
        -  6 * work['n_weaknesses']
    )
    req  = work[work['is_required_type']].nlargest(POOL_REQUIRED, 'solo_score')
    other = work[~work['is_required_type']].nlargest(POOL_OTHER,    'solo_score')

    if len(req) < REQUIRED_COUNT or len(other) < (TEAM_SIZE - REQUIRED_COUNT):
        raise ValueError("Pas assez de candidats pour satisfaire la contrainte.")

    return work, req, other

## Team score

In [ ]:
def team_score(team):
    # Couverture offensive
    coverage = set()
    for cov in team['offense_coverage']:
        coverage |= set(cov) if isinstance(cov, (set, list, tuple)) else set()

    # Profils défensifs
    mult = np.array([
        [prof.get(t, 1.0) if isinstance(prof, dict) else 1.0 for t in ALL_TYPES]
        for prof in team['defense_profile']
    ], dtype=float)

    weak_counts   = (mult > 1.0).sum(axis=0)
    resist_counts = (mult < 1.0).sum(axis=0)
    immune_counts = (mult == 0.0).sum(axis=0)
    holes         = (mult.min(axis=0) > 1.0).sum()
    stacked_weak  = (weak_counts >= 3).sum()

    return (
        2.8 * len(coverage)
        + 0.09 * team['bst'].sum()
        + 0.9  * team['defense_score'].sum()
        + 1.8  * (resist_counts >= 2).sum()
        + 2.2  * (immune_counts >= 1).sum()
        - 3.0  * holes
        - 1.3  * stacked_weak
        + 0.8  * team['type1'].nunique()
    )

## Trouver le meilleurs score

In [3]:
def search_best_team(work, req_pool, other_pool):
    rng     = np.random.default_rng(SEED)
    req_idx = req_pool.index.to_numpy()
    oth_idx = other_pool.index.to_numpy()
    best_score, best_team = -np.inf, None

    for _ in range(N_ITER):
        idx = np.concatenate([
            rng.choice(req_idx, size=REQUIRED_COUNT,            replace=False),
            rng.choice(oth_idx, size=TEAM_SIZE - REQUIRED_COUNT, replace=False),
        ])
        team = work.loc[idx]
        s    = team_score(team)
        if s > best_score:
            best_score, best_team = s, team.copy()

    return best_score, best_team

def display_results(best_team, best_score):
    best_team = best_team.assign(
        is_water=best_team['types'].apply(lambda x: has_type(x, REQUIRED_TYPE))
    ).sort_values(['is_water', 'bst'], ascending=[False, False])

    print(f"Meilleur score    : {best_score:.2f}")
    print(f"Pokémon {REQUIRED_TYPE:8s} : {best_team['is_water'].sum()}/{TEAM_SIZE}")
    print("Équipe            :", ", ".join(best_team['name'].tolist()))

    display(best_team[[
        'name', 'types', 'bst', 'hp', 'attack', 'defense',
        'special-attack', 'special-defense', 'speed',
        'defense_score', 'n_weaknesses', 'n_resistances', 'n_immunities', 'n_offense'
    ]].reset_index(drop=True))

In [ ]:
team_cov = set().union(*[
    set(cov) if isinstance(cov, (set, list, tuple)) else set()
    for cov in best_team['offense_coverage']
])

display(best_team[[
    'name', 'type1', 'type2', 'hp', 'attack', 'defense',
    'special-attack', 'special-defense', 'speed', 'bst'
]].reset_index(drop=True))
print(f"Score équipe: {best_score:.1f} | Couverture offensive: {len(team_cov)}/{len(ALL_TYPES)} types")

stats_df = best_team.set_index('name')[STATS].rename(columns={
    'hp': 'HP',
    'attack': 'ATK',
    'defense': 'DEF',
    'special-attack': 'SPA',
    'special-defense': 'SPD',
    'speed': 'SPE',
})

labels = stats_df.columns.tolist()
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

max_stat = max(100, int(np.ceil(stats_df.to_numpy().max() / 25) * 25))
ticks = np.arange(50, max_stat + 1, 50)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'polar': True})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_ylim(0, max_stat)
ax.set_yticks(ticks)
ax.set_yticklabels([str(t) for t in ticks])

for i, (name, row) in enumerate(stats_df.iterrows()):
    values = row.tolist() + [row.iloc[0]]
    color = TEAM_COLORS[i % len(TEAM_COLORS)]
    ax.plot(angles, values, color=color, linewidth=2, label=name)
    ax.fill(angles, values, color=color, alpha=0.12)

ax.set_title(f"Radar des statistiques — Meilleure équipe ({REQUIRED_COUNT} {REQUIRED_TYPE})", pad=18)
ax.legend(loc='upper right', bbox_to_anchor=(1.30, 1.10), frameon=False)
plt.tight_layout()
plt.show()